# 🖼️ DigitVision: Handwritten Digit Classification

## 📌 Project Overview
**DigitVision** is a PyTorch-based Deep Learning project designed to classify handwritten digits (0–9) using a custom **Convolutional Neural Network (CNN)** trained on the classic **MNIST dataset**.

---

## 🛠️ Tech Stack & Dependencies
- **Framework:** PyTorch (`torch`, `torch.nn`, `torch.optim`)
- **Computer Vision Library:** `torchvision` (`datasets`, `transforms`)
- **Hardware Acceleration:** CUDA / GPU (if available)

---

## 🏗️ Model Architecture
The custom CNN consists of 3 convolutional blocks followed by fully connected dense layers:
1. **Conv Block 1:** `Conv2d` (1 $\rightarrow$ 32 filters, $3\times3$) + `ReLU` + `MaxPool2d` ($2\times2$)
2. **Conv Block 2:** `Conv2d` (32 $\rightarrow$ 64 filters, $3\times3$) + `ReLU` + `MaxPool2d` ($2\times2$)
3. **Conv Block 3:** `Conv2d` (64 $\rightarrow$ 128 filters, $3\times3$) + `ReLU` + `MaxPool2d` ($2\times2$)
4. **Classifier:** Flatten $\rightarrow$ `Linear` (1152 $\rightarrow$ 256) $\rightarrow$ `ReLU` $\rightarrow$ `Linear` (256 $\rightarrow$ 10 outputs)

---

## ⚙️ Training Hyperparameters
- **Batch Size:** 64
- **Optimizer:** Adam (`lr=0.001`)
- **Loss Function:** CrossEntropyLoss
- **Epochs:** 10

---

## 📊 Key Results
- **Final Training Loss:** `0.008`
- **Test Set Accuracy:** `98.72%`

In [2]:
import torch 
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [3]:
# Device configuration (GPU if available, else CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = datasets.MNIST(root='./Digit',train=True,download=True,transform=transform)
testset = datasets.MNIST(root='./Digit',train=False,download=True,transform=transform)

In [5]:
trainset

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./Digit
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5,), std=(0.5,))
           )

In [6]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64,shuffle=False)

In [7]:
# Building the CNN
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv_layers = nn.Sequential(
            # layer 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # layer 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # layer 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(3 * 3 * 128, 256),
            nn.ReLU(),
            nn.Linear(256, 10) # 10 outputs representing digit classes (0-9)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # Flatten tensor
        x = self.fc_layer(x)
        return x

model = CNN().to(device)

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [9]:
# Training the model
epochs = 10

for epoch in range(epochs):
    running_loss = 0.0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(trainloader):.4f}")

Epoch 1/10 - Loss: 0.1518
Epoch 2/10 - Loss: 0.0424
Epoch 3/10 - Loss: 0.0292
Epoch 4/10 - Loss: 0.0233
Epoch 5/10 - Loss: 0.0175
Epoch 6/10 - Loss: 0.0149
Epoch 7/10 - Loss: 0.0138
Epoch 8/10 - Loss: 0.0106
Epoch 9/10 - Loss: 0.0091
Epoch 10/10 - Loss: 0.0098


In [10]:
# Evaluation
correct_labels = 0
total_labels = 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy: {(correct_labels / total_labels) * 100:.2f}%")
    

Accuracy: 98.72%


In [11]:
# Save the trained model for the UI
checkpoint = {
    "model_state_dict": model.state_dict(),
    "accuracy": (correct_labels / total_labels) * 100,
}

torch.save(checkpoint, "digitvision_model.pth")

print("Model saved successfully as digitvision_model.pth")

Model saved successfully as digitvision_model.pth
